# JUPITER NOTEBOOK - EXPLORACION
Aqui se explorara los datos 

In [1]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df_transaction = pd.read_csv('../data/transactions_data.csv')
df_transaction.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN


In [3]:
df_user = pd.read_csv('../data/users_data.csv')
df_user.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [4]:
df_cards = pd.read_csv('../data/cards_data.csv')
df_cards.head()

,id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


In [5]:
with open('../data/train_fraud_labels.json') as f:
    fraud_labels = json.load(f)

fraude_inner = fraud_labels['target']
df_fraud_labels = pd.DataFrame(list(fraude_inner.items()), columns=['id', 'is_fraud'])
df_fraud_labels.head()

,id,is_fraud
0,10649266,No
1,23410063,No
2,9316588,No
3,12478022,No
4,9558530,No


In [6]:
with open('../data/mcc_codes.json') as f:
    mcc_codes = json.load(f)

df_mcc_codes = pd.DataFrame(list(mcc_codes.items()), columns=['mcc_code', 'categoria'])
df_mcc_codes.head()

,mcc_code,categoria
0,5812,Eating Places and Restaurants
1,5541,Service Stations
2,7996,"Amusement Parks, Carnivals, Circuses"
3,5411,"Grocery Stores, Supermarkets"
4,4784,Tolls and Bridge Fees


Hacer un merge con df_transaction y df_mcc_codes

In [7]:
df_mcc_codes.dtypes

mcc_code     object
categoria    object
dtype: object

In [8]:
df_mcc_codes['mcc_code'] = df_mcc_codes['mcc_code'].astype(int)
df_mcc_codes.dtypes

mcc_code      int64
categoria    object
dtype: object

In [9]:
df_fraud_labels.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8914963 entries, 0 to 8914962
Data columns (total 2 columns):
 #   Column    Dtype 
---  ------    ----- 
 0   id        object
 1   is_fraud  object
dtypes: object(2)
memory usage: 136.0+ MB


In [10]:
df_mcc_codes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 109 entries, 0 to 108
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   mcc_code   109 non-null    int64 
 1   categoria  109 non-null    object
dtypes: int64(1), object(1)
memory usage: 1.8+ KB


In [11]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   object 
 6   address            2000 non-null   object 
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   object 
 10  yearly_income      2000 non-null   object 
 11  total_debt         2000 non-null   object 
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 218.9+ KB


In [12]:
df_transaction = df_transaction.merge(df_mcc_codes, left_on='mcc', right_on='mcc_code', how='left')
df_transaction.head()

,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,mcc_code,categoria
0,7475327,2010-01-01 00:01:00,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523.0,5499,NaN,5499,Miscellaneous Food Stores
1,7475328,2010-01-01 00:02:00,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722.0,5311,NaN,5311,Department Stores
2,7475329,2010-01-01 00:02:00,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084.0,4829,NaN,4829,Money Transfer
3,7475331,2010-01-01 00:05:00,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307.0,4829,NaN,4829,Money Transfer
4,7475332,2010-01-01 00:06:00,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776.0,5813,NaN,5813,Drinking Places (Alcoholic Beverages)


### Pregunta 1 - ¿Cuántas transacciones tiene el dataset y cuántas columnas?
- Transacciones: 13,305,915
- Columnas: 14

In [13]:
df_transaction.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13305915 entries, 0 to 13305914
Data columns (total 14 columns):
 #   Column          Dtype  
---  ------          -----  
 0   id              int64  
 1   date            object 
 2   client_id       int64  
 3   card_id         int64  
 4   amount          object 
 5   use_chip        object 
 6   merchant_id     int64  
 7   merchant_city   object 
 8   merchant_state  object 
 9   zip             float64
 10  mcc             int64  
 11  errors          object 
 12  mcc_code        int64  
 13  categoria       object 
dtypes: float64(1), int64(6), object(7)
memory usage: 1.4+ GB


In [14]:
df_transaction.dtypes

id                  int64
date               object
client_id           int64
card_id             int64
amount             object
use_chip           object
merchant_id         int64
merchant_city      object
merchant_state     object
zip               float64
mcc                 int64
errors             object
mcc_code            int64
categoria          object
dtype: object

### Pregunta 2 - ¿Hay valores nulos?¿En qué columnas?
- merchant_state:     1,563,700
- zip:                1,652,706
- errors:            13,094,522

In [15]:
df_transaction.isnull().sum()

id                       0
date                     0
client_id                0
card_id                  0
amount                   0
use_chip                 0
merchant_id              0
merchant_city            0
merchant_state     1563700
zip                1652706
mcc                      0
errors            13094522
mcc_code                 0
categoria                0
dtype: int64

In [16]:
df_transaction['errors'] = df_transaction['errors'].fillna("Sin Error")
print("Valores nulos en 'errors': ", df_transaction['errors'].isnull().sum())

Valores nulos en 'errors':  0


In [17]:
print("Merchant State null values:", df_transaction[df_transaction['merchant_state'].isnull()]['use_chip'].value_counts())
print("Zip null values:", df_transaction[df_transaction['zip'].isnull()]['use_chip'].value_counts())

Merchant State null values: use_chip
Online Transaction    1557912
Chip Transaction         5788
Name: count, dtype: int64
Zip null values: use_chip
Online Transaction    1557912
Swipe Transaction       48493
Chip Transaction        46301
Name: count, dtype: int64


In [18]:
total = len(df_transaction)
merchant_state_null = df_transaction['merchant_state'].isnull().sum()
zip_null = df_transaction['zip'].isnull().sum()

In [19]:
print(f"Porcentaje de valores nulos en 'merchant_state': {merchant_state_null / total * 100:.2f}%")
print(f"Porcentaje de valores nulos en 'zip': {zip_null / total * 100:.2f}%")

Porcentaje de valores nulos en 'merchant_state': 11.75%
Porcentaje de valores nulos en 'zip': 12.42%


In [20]:
df_transaction['use_chip'].value_counts()

use_chip
Swipe Transaction     6967185
Chip Transaction      4780818
Online Transaction    1557912
Name: count, dtype: int64

In [21]:
mask_online = df_transaction['use_chip'] == 'Online Transaction'
df_transaction.loc[mask_online & df_transaction['merchant_state'].isnull(), 'merchant_state'] = 'Online'

# Rellenar el resto con 'Desconocido'
df_transaction['merchant_state'] = df_transaction['merchant_state'].fillna('Desconocido')

# Verificar que no quedaron nulos
print(df_transaction['merchant_state'].isnull().sum())

0


In [22]:
df_transaction['zip'] = df_transaction['zip'].astype(str)

df_transaction.loc[mask_online & df_transaction['zip'].isin(['nan', 'NaN']), 'zip'] = 'Online'
# Rellenar el resto con 'Desconocido'
df_transaction['zip'] = df_transaction['zip'].replace('nan', 'Desconocido')
# Verificar que no quedaron nulos
print(df_transaction['zip'].isnull().sum())

0


In [23]:
df_transaction.isnull().sum()

id                0
date              0
client_id         0
card_id           0
amount            0
use_chip          0
merchant_id       0
merchant_city     0
merchant_state    0
zip               0
mcc               0
errors            0
mcc_code          0
categoria         0
dtype: int64

### Pregunta 3 - ¿Cuál es el monto mínimo, máximo, promedio y mediana de las transacciones?
- Valor Mínimo: -500
- Valor Máximo: 6,820.2
- Promedio: 42.98
- Mediana: 29.99

In [24]:
#Cambiar el tipo de datos de df_transaction["amount"] para poder calcular el valor mínimo, máximo, promedio y mediana
df_transaction["amount"] = df_transaction["amount"].str.replace("$","", regex=False).astype(float)

In [30]:
print("Mínimo", df_transaction["amount"].min())
print("Máximo", df_transaction["amount"].max())
print("Promedio: ", round(df_transaction["amount"].mean(),2))
print("Mediana", df_transaction["amount"].median())

Mínimo -500.0
Máximo 6820.2
Promedio:  42.98
Mediana 28.99


### Pregunta 4 - ¿En qué rango de fechas están los datos?

In [32]:
df_transaction["date"] = pd.to_datetime(df_transaction["date"])
min_date = df_transaction["date"].min()
max_date = df_transaction['date'].max()

print(f"Fecha mínima: {min_date}")
print(f"Fecha máxima: {max_date}")

Fecha mínima: 2010-01-01 00:01:00
Fecha máxima: 2019-10-31 23:59:00


### Pregunta 5 - ¿Cuánta transacciones únicos por tipo o categoría?
- 3 Tipos de transacciones
  - Swipe Transaction (52.36%)
  - Chip Transaction (35.93%)
  - Online Transaction (11.71%)
- 108 Categorias
    - Top 3
        -   Grocery Stores, Supermarkets
        -   Miscellaneous Food Stores
        -   Service Stations
    - Buton 3
        - Bolt, Nut, Screw, Rivet Manufacturing
        - Floor Covering Stores
        - Music Stores - Musical Instruments  

In [33]:
df_transaction["use_chip"].value_counts()

use_chip
Swipe Transaction     6967185
Chip Transaction      4780818
Online Transaction    1557912
Name: count, dtype: int64

In [43]:
print(round(df_transaction["use_chip"].value_counts(normalize=True)*100,2))

use_chip
Swipe Transaction     52.36
Chip Transaction      35.93
Online Transaction    11.71
Name: proportion, dtype: float64


In [35]:
print("Cantidad de Categorias: ",df_transaction["categoria"].nunique())

Cantidad de Categorias:  108


In [41]:
conteo_categoria = df_transaction["categoria"].value_counts().sort_values(ascending=False)

print("Top 3", conteo_categoria.head(3))
print("Buton 3", conteo_categoria.tail(3))

Top 3 categoria
Grocery Stores, Supermarkets    1592584
Miscellaneous Food Stores       1460875
Service Stations                1424711
Name: count, dtype: int64
Buton 3 categoria
Bolt, Nut, Screw, Rivet Manufacturing    337
Floor Covering Stores                    334
Music Stores - Musical Instruments       319
Name: count, dtype: int64


### Mas Exploración

In [26]:
df_transaction.describe()

,id,client_id,card_id,amount,merchant_id,mcc,mcc_code
count,1.330592e+07,1.330592e+07,1.330592e+07,1.330592e+07,1.330592e+07,1.330592e+07,1.330592e+07
mean,1.558402e+07,1.026812e+03,3.475268e+03,4.297604e+01,4.772376e+04,5.565440e+03,5.565440e+03
std,4.704499e+06,5.816386e+02,1.674356e+03,8.165575e+01,2.581534e+04,8.757002e+02,8.757002e+02
min,7.475327e+06,0.000000e+00,0.000000e+00,-5.000000e+02,1.000000e+00,1.711000e+03,1.711000e+03
25%,1.150604e+07,5.190000e+02,2.413000e+03,8.930000e+00,2.588700e+04,5.300000e+03,5.300000e+03
50%,1.557087e+07,1.070000e+03,3.584000e+03,2.899000e+01,4.592600e+04,5.499000e+03,5.499000e+03
75%,1.965361e+07,1.531000e+03,4.901000e+03,6.371000e+01,6.757000e+04,5.812000e+03,5.812000e+03
max,2.376187e+07,1.998000e+03,6.144000e+03,6.820200e+03,1.003420e+05,9.402000e+03,9.402000e+03


In [27]:
df_transaction['amount'].describe()

count    1.330592e+07
mean     4.297604e+01
std      8.165575e+01
min     -5.000000e+02
25%      8.930000e+00
50%      2.899000e+01
75%      6.371000e+01
max      6.820200e+03
Name: amount, dtype: float64

In [48]:
df_user.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 2000 non-null   int64  
 1   current_age        2000 non-null   int64  
 2   retirement_age     2000 non-null   int64  
 3   birth_year         2000 non-null   int64  
 4   birth_month        2000 non-null   int64  
 5   gender             2000 non-null   object 
 6   address            2000 non-null   object 
 7   latitude           2000 non-null   float64
 8   longitude          2000 non-null   float64
 9   per_capita_income  2000 non-null   object 
 10  yearly_income      2000 non-null   object 
 11  total_debt         2000 non-null   object 
 12  credit_score       2000 non-null   int64  
 13  num_credit_cards   2000 non-null   int64  
dtypes: float64(2), int64(7), object(5)
memory usage: 218.9+ KB


In [47]:
df_user.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


In [49]:
df_user['yearly_income'] = df_user['yearly_income'].str.replace("$","", regex=False).astype(int)
df_user['total_debt'] = df_user['total_debt'].str.replace("$","", regex=False).astype(int)
df_user

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,59696,127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,77254,191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,33483,196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,249925,202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,109687,183855,675,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,986,32,70,1987,7,Male,6577 Lexington Lane,40.65,-73.58,$23550,48010,87837,703,3
1996,1944,62,65,1957,11,Female,2 Elm Drive,38.95,-84.54,$24218,49378,104480,740,4
1997,185,47,67,1973,1,Female,276 Fifth Boulevard,40.66,-74.19,$15175,30942,71066,779,3
1998,1007,66,60,1954,2,Male,259 Valley Boulevard,40.24,-76.92,$25336,54654,27241,618,1


In [50]:
df_user['per_capita_income'] = df_user['per_capita_income'].str.replace("$","", regex=False).astype(int)
df_user.head()

,id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278,59696,127613,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891,77254,191349,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681,33483,196,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145,249925,202328,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797,109687,183855,675,1


In [51]:
df_user.to_csv(r"C:\Users\Personal\Documents\PythonDataAnalisis\Portafolio\Python_Projects\Analisis Financiero\data\df_user_clean.csv", index=False)

In [46]:
df_transaction.to_csv(r"C:\Users\Personal\Documents\PythonDataAnalisis\Portafolio\Python_Projects\Analisis Financiero\data\df_transaction_clean.csv", index=False)